## Refresh Excel Pipeline

Functions in this notebook query DynamoDB records (Accounts, Instruments, and Holdings) and store results in local CSV files. 

The CSV files are dynamically accessed by [Excel workbook](/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/etrade-equity-portfolio.xlsx) using Power Query.



In [4]:
from datetime import datetime
today_str = datetime.now().strftime("%Y-%m-%d")
win_outpath = "/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current"
local_outpath = win_outpath

print(f"Today's date prefix is: {today_str}")
print(f"Windows output path is: {win_outpath}")

%reload_ext autoreload
%autoreload 2

from ramlytics_database import DynamoDBClient
from ramlytics_database import DbQuery
#from ramlytics_yfinance import yf_search

qry = DbQuery(DynamoDBClient())

def store_csv(outpath, df, filename):
    import os
    from datetime import datetime
    today_str = datetime.today().strftime('%Y-%m-%d')

    df.to_csv(f"{outpath}/{filename}", index=False)
    # Save a copy of the CSV in the archive folder with today's date
    archive = outpath.replace("current", f"archive/{today_str}")
    os.makedirs(archive, exist_ok=True)
    df.to_csv(f"{archive}/{filename}", index=False)

    return f"{outpath}/{filename}", f"{archive}/{filename}"

def refresh_instruments(symbols = []):
    if not symbols:
        symbols = qry.instruments_df['symbol'].tolist()
    for symbol in symbols:
        inst = qry.db.get_instrument(symbol)
        inst.pop("symbol")
        response = qry.db.put_instrument(symbol, **inst)  

    qry.get_instruments_df()
      
    return f"refreshed symbols: {symbols}, response: {response}"

def refresh_all_csv(outpath="."):
    refresh_instruments()
    qry = DbQuery(DynamoDBClient())    

    instrument_holdings = qry.get_instrument_holdings_df()
    positions = qry.get_account_positions_df()
    position_holdings = qry.get_position_holdings(instrument_holdings, positions)

    store_csv(outpath, df=qry.get_instrument_market_summary_df(), filename="market_summary.csv")
    store_csv(outpath, df=qry.get_account_headers_df(), filename="account_headers.csv")
    store_csv(outpath, df=qry.get_acct_summary_by_symbol_df(), filename="acct_summary_by_symbol.csv")
    store_csv(outpath, df=instrument_holdings, filename="instrument_holdings.csv")
    store_csv(outpath, df=positions, filename="account_positions.csv")
    store_csv(outpath, df=position_holdings, filename="position_holdings.csv")
    store_csv(outpath, df=qry.get_eft_holdings_by_secname(position_holdings), filename="eft_holdings_by_secname.csv")
    store_csv(outpath, df=qry.get_instrument_scores_df(), filename="instrument_scores.csv")

    return

refresh_all_csv(outpath=win_outpath)

print("CSV refresh complete.")
print(f"Files stored in: {win_outpath}")



Today's date prefix is: 2026-11-11
Windows output path is: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current


INFO:root:{'exchange': 'NMS', 'shortname': 'Apple Inc.', 'quoteType': 'EQUITY', 'symbol': 'AAPL', 'index': 'quotes', 'score': 235334000.0, 'typeDisp': 'Equity', 'longname': 'Apple Inc.', 'exchDisp': 'NASDAQ', 'sector': 'Technology', 'sectorDisp': 'Technology', 'industry': 'Consumer Electronics', 'industryDisp': 'Consumer Electronics', 'dispSecIndFlag': True, 'isYahooFinance': True, 'fastinfo': {'currency': 'USD', 'dayHigh': 354.44000244140625, 'dayLow': 349.32501220703125, 'exchange': 'NMS', 'fiftyDayAverage': 337.2668011474609, 'lastPrice': 353.3999938964844, 'lastVolume': 7866966, 'marketCap': 5135673030102.6, 'open': 349.5849914550781, 'previousClose': 348.81, 'quoteType': 'EQUITY', 'regularMarketPreviousClose': 348.510009765625, 'shares': 14532182000, 'tenDayAverageVolume': 56254740, 'threeMonthAverageVolume': 42739044, 'timezone': 'America/New_York', 'twoHundredDayAverage': 299.7839501953125, 'yearChange': 0.2743994156944515, 'yearHigh': 355.3800048828125, 'yearLow': 243.419998168

CSV refresh complete.
Files stored in: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/excel_pipeline/current


In [ ]:
qry = DbQuery(DynamoDBClient())    

instrument_holdings = qry.get_instrument_holdings_df()

INFO:root:{'exchange': 'NMS', 'shortname': 'Apple Inc.', 'quoteType': 'EQUITY', 'symbol': 'AAPL', 'index': 'quotes', 'score': 235334000.0, 'typeDisp': 'Equity', 'longname': 'Apple Inc.', 'exchDisp': 'NASDAQ', 'sector': 'Technology', 'sectorDisp': 'Technology', 'industry': 'Consumer Electronics', 'industryDisp': 'Consumer Electronics', 'dispSecIndFlag': True, 'isYahooFinance': True, 'fastinfo': {'currency': 'USD', 'dayHigh': 354.44000244140625, 'dayLow': 349.32501220703125, 'exchange': 'NMS', 'fiftyDayAverage': 337.2668011474609, 'lastPrice': 353.5050048828125, 'lastVolume': 7038740, 'marketCap': 5137199068867.92, 'open': 349.5849914550781, 'previousClose': 348.81, 'quoteType': 'EQUITY', 'regularMarketPreviousClose': 348.510009765625, 'shares': 14532182000, 'tenDayAverageVolume': 56254740, 'threeMonthAverageVolume': 42739044, 'timezone': 'America/New_York', 'twoHundredDayAverage': 299.7839501953125, 'yearChange': 0.2743994156944515, 'yearHigh': 355.3800048828125, 'yearLow': 243.41999816

In [ ]:

instrument_holdings = qry.get_instrument_holdings_df()